In [1]:
import pandas as pd
import numpy as np

from utilities import get_path, read_data_from_database

In [2]:
pd.options.display.max_columns = None  # Remove "dots" from display when printing dataframes

In [3]:
PATH = get_path(1)
DATABASE_FOLDER = PATH + 'data/preprocessing/clear_data.db'

In [4]:
df = read_data_from_database(DATABASE_FOLDER, 'train_table')

df.head()

,ID,Edad,Tipo_Trabajo,Estado_Civil,Educacion,mora,Vivienda,Consumo,Contacto,Mes,Dia,Campana,Dias_Ultima_Camp,No_Contactos,Resultado_Anterior,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,1,57,servicios,casado,bachillerato,NaN,0.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
1,2,37,servicios,casado,bachillerato,0.0,1.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
2,3,40,administrador negocio,casado,primaria,0.0,0.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
3,4,56,servicios,casado,bachillerato,0.0,0.0,1.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
4,7,25,servicios,soltero,bachillerato,0.0,1.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0


In [5]:
df.isna().sum()

ID                       0
Edad                     0
Tipo_Trabajo           165
Estado_Civil            39
Educacion              958
mora                  4766
Vivienda               553
Consumo                553
Contacto                 0
Mes                      0
Dia                      0
Campana                  0
Dias_Ultima_Camp         0
No_Contactos             0
Resultado_Anterior       0
emp_var_rate             0
cons_price_idx           0
cons_conf_idx            0
euribor3m                0
nr_employed              0
y                        0
dtype: int64

# Datos faltantes

## Variables categóricas

In [6]:
cols_relev = ['mora', 'Vivienda', 'Consumo', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m']
cols_complet = cols_relev.copy()
cols_complet.append('Tipo_Trabajo')

print(df.shape)
df_homog = df.dropna().copy()
print(df_homog.shape)

# 1. Crear dataset de entrenamiento
# 1.1 Extraer filas con datos completos
XY = df_homog.dropna()[cols_complet].to_numpy().copy()
# print(XY.shape)

# 1.2 Crear el set de entrenamiento
x_train = XY[:,0:len(cols_complet)-2]
y_train = XY[:,len(cols_complet)-1]  # Tipo_Trabajo

# 2. Crear dataset de prueba: filas con datos incompletos
rows = list(df[~df['Tipo_Trabajo'].notna()].index)  # Incompletd rows
x_test = df[cols_relev].iloc[rows].to_numpy()

# 3. Escoger y entrenar el modelo de ML con el set de entrenamiento
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

le = LabelEncoder()
le.fit(y_train)
# le.classes_
y_train = le.transform(y_train)
# le.inverse_transform([0,1,2,3])  # Ejemplos al azar
lr = LogisticRegression()  # Instancia del modelo
lr.fit(x_train, y_train)

(23099, 21)
(17176, 21)


d:\Proyectos\Pruebas técnicas\Banco de Bogotá\venv\lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [7]:
x_test

array([[        nan,  1.0000e+00,  0.0000e+00, ...,  9.3994e+04,
        -3.6400e+01,  4.8570e+03],
       [        nan,  0.0000e+00,  0.0000e+00, ...,  9.3994e+04,
        -3.6400e+01,  4.8570e+03],
       [        nan,  1.0000e+00,  1.0000e+00, ...,  9.3994e+04,
        -3.6400e+01,  4.8570e+03],
       ...,
       [        nan,  1.0000e+00,  0.0000e+00, ...,  9.4199e+04,
        -3.7500e+01,  8.8000e-01],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  9.4601e+04,
        -4.9500e+01,  1.0250e+03],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00, ...,  9.4767e+04,
        -5.0800e+01,  1.0480e+03]])

In [8]:
# Predictions
preds = lr.predict(x_test)
cats = le.inverse_transform(preds)

# Completar dataframe con categorías predichas
df_ml = df.copy()
df_ml.iloc[[1,2], 2] = cats  # 'Tipo_Trabajo' es la columna 2
df_ml

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# df[rows,2]
rows

Index([   76,   156,   201,   228,   260,   274,   316,   418,   462,   628,
       ...
       22287, 22291, 22350, 22403, 22436, 22528, 22654, 22797, 22993, 23049],
      dtype='int64', length=165)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23099 entries, 0 to 23098
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  23099 non-null  int64  
 1   Edad                23099 non-null  int64  
 2   Tipo_Trabajo        22934 non-null  object 
 3   Estado_Civil        23060 non-null  object 
 4   Educacion           22141 non-null  object 
 5   mora                18333 non-null  float64
 6   Vivienda            22546 non-null  float64
 7   Consumo             22546 non-null  float64
 8   Contacto            23099 non-null  object 
 9   Mes                 23099 non-null  object 
 10  Dia                 23099 non-null  object 
 11  Campana             23099 non-null  object 
 12  Dias_Ultima_Camp    23099 non-null  int64  
 13  No_Contactos        23099 non-null  int64  
 14  Resultado_Anterior  23099 non-null  object 
 15  emp_var_rate        23099 non-null  float64
 16  cons

## Variables numéricas